# ⚽ Predict the FIFA World Cup 2026

## 📖 Background

The 2026 FIFA World Cup is one of the biggest sporting events in the world, hosted across the United States, Canada, and Mexico. For the first time, the tournament expands to 48 teams, producing 104 matches across the group stage and knockout rounds.

Using machine learning, historical statistics, and soccer domain knowledge, predict match scores, corners, and cards for every fixture. You must submit all your predictions before a single ball is kicked.

The scoring system rewards precision: an exact scoreline earns maximum points, while close predictions still earn partial credit. Later rounds carry score multipliers, so a strong model that holds up in the knockout stages can leapfrog the competition. The challenge is designed to be difficult enough that no one can achieve a perfect score—even with AI assistance—but accessible enough that any data enthusiast can participate and score points.

## 💾 The data

You have access to the following files:

#### `data/group_fixtures.csv` — all 72 group stage matches
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `group` | Group letter (A–L) |
| `home_team` | Home team name |
| `away_team` | Away team name |
| `date` | Match date (UTC) |
| `venue` | Stadium and city |

#### `data/knockout_slots.csv` — all 32 knockout round slots
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `round` | Round name (e.g. `Quarter-final`) |
| `multiplier` | Score multiplier for this round |
| `slot_home` | Description of the home team slot (e.g. `Winner Group A`) |
| `slot_away` | Description of the away team slot |

| Variable | Description |
|---|---|

You may also bring in any external data—FIFA rankings, historical match results, player statistics—to build your predictions.

# ═══════════════════════════════════════════════════════════════════════════════
# FIFA WORLD CUP 2026 — COMPLETE PREDICTION SYSTEM
# ─────────────────────────────────────────────────
# Author    : iTaqiZ | Pakistan
# Version   : 3.0 (Squad-informed Bivariate Poisson + Monte Carlo Bracket)
# Data      : Official FIFA Schedule PDF (Apr 10 2026) · Squad Lists (Jun 3 2026)
#             Kaggle WC2026 Baseline · FIFA Rankings · Transfermarkt values
# Predicted champion : FRANCE 🏆
# ═══════════════════════════════════════════════════════════════════════════════

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import poisson
import warnings
warnings.filterwarnings('ignore')

np.random.seed(2026)

In [2]:
# ── CELL 1 · DATA LOADING ──────────────────────────────────────────────────────

try:
    group_fixtures = pd.read_csv('data/group_fixtures.csv')
    knockout_slots  = pd.read_csv('data/knockout_slots.csv')
    print(f"✓ DataLab CSV loaded — {len(group_fixtures)} group fixtures | "
          f"{len(knockout_slots)} knockout slots")
    FROM_CSV = True
except FileNotFoundError:
    FROM_CSV = False
    print("▶  CSV not found — constructing from official FIFA schedule")

if not FROM_CSV:
    # ── Official FIFA Match Schedule (Source: FIFA PDF v17, April 10 2026) ──────
    # All 72 group stage matches with official match IDs, home/away teams, dates

    RAW = [
        # (match_id, group, home_team, away_team, date, venue)
        # GROUP A — Mexico · South Africa · Korea Republic · Czechia
        ( 1,'A','Mexico',                    'South Africa',            '2026-06-11','Estadio Azteca, Mexico City'),
        ( 2,'A','Korea Republic',            'Czechia',                 '2026-06-12','Estadio Akron, Guadalajara'),
        (25,'A','Czechia',                   'South Africa',            '2026-06-18','Mercedes-Benz Stadium, Atlanta'),
        (27,'A','Mexico',                    'Korea Republic',          '2026-06-19','Estadio Azteca, Mexico City'),
        (51,'A','South Africa',              'Korea Republic',          '2026-06-25','Estadio BBVA, Monterrey'),
        (53,'A','Czechia',                   'Mexico',                  '2026-06-25','Estadio Azteca, Mexico City'),
        # GROUP B — Canada · Bosnia and Herzegovina · Qatar · Switzerland
        ( 3,'B','Canada',                    'Bosnia and Herzegovina',  '2026-06-12','BMO Field, Toronto'),
        ( 8,'B','Qatar',                     'Switzerland',             '2026-06-13',"Levi's Stadium, Santa Clara"),
        (26,'B','Switzerland',               'Bosnia and Herzegovina',  '2026-06-18','SoFi Stadium, Los Angeles'),
        (44,'B','Canada',                    'Qatar',                   '2026-06-19','BC Place, Vancouver'),
        (52,'B','Bosnia and Herzegovina',    'Qatar',                   '2026-06-25','Lumen Field, Seattle'),
        (70,'B','Switzerland',               'Canada',                  '2026-06-25','BC Place, Vancouver'),
        # GROUP C — Brazil · Morocco · Haiti · Scotland
        ( 7,'C','Brazil',                    'Morocco',                 '2026-06-13','MetLife Stadium, East Rutherford'),
        ( 5,'C','Haiti',                     'Scotland',                '2026-06-14','Gillette Stadium, Foxborough'),
        (28,'C','Brazil',                    'Haiti',                   '2026-06-20','Lincoln Financial Field, Philadelphia'),
        (29,'C','Scotland',                  'Morocco',                 '2026-06-19','Gillette Stadium, Foxborough'),
        (47,'C','Morocco',                   'Haiti',                   '2026-06-26','MetLife Stadium, East Rutherford'),
        (50,'C','Scotland',                  'Brazil',                  '2026-06-26','Gillette Stadium, Foxborough'),
        # GROUP D — USA · Paraguay · Australia · Türkiye
        ( 4,'D','United States',             'Paraguay',                '2026-06-12','SoFi Stadium, Los Angeles'),
        ( 6,'D','Australia',                 'Türkiye',                 '2026-06-13','AT&T Stadium, Arlington'),
        (30,'D','United States',             'Australia',               '2026-06-19','SoFi Stadium, Los Angeles'),
        (31,'D','Türkiye',                   'Paraguay',                '2026-06-18','Hard Rock Stadium, Miami'),
        (58,'D','Paraguay',                  'Australia',               '2026-06-26','Commanders Field, Washington DC'),
        (59,'D','Türkiye',                   'United States',           '2026-06-26','AT&T Stadium, Arlington'),
        # GROUP E — Germany · Curaçao · Côte d'Ivoire · Ecuador
        (10,'E','Germany',                   'Curaçao',                 '2026-06-14','MetLife Stadium, East Rutherford'),
        ( 9,'E',"Côte d'Ivoire",             'Ecuador',                 '2026-06-14','Arrowhead Stadium, Kansas City'),
        (34,'E','Germany',                   "Côte d'Ivoire",           '2026-06-20','Gillette Stadium, Foxborough'),
        (36,'E','Ecuador',                   'Curaçao',                 '2026-06-19','Rose Bowl, Pasadena'),
        (49,'E','Ecuador',                   'Germany',                 '2026-06-25','MetLife Stadium, East Rutherford'),
        (56,'E','Curaçao',                   "Côte d'Ivoire",           '2026-06-25','Arrowhead Stadium, Kansas City'),
        # GROUP F — Netherlands · Japan · Sweden · Tunisia
        (11,'F','Netherlands',               'Japan',                   '2026-06-15','Lincoln Financial Field, Philadelphia'),
        (12,'F','Sweden',                    'Tunisia',                 '2026-06-15','Lumen Field, Seattle'),
        (32,'F','Tunisia',                   'Japan',                   '2026-06-20','Lumen Field, Seattle'),
        (35,'F','Netherlands',               'Sweden',                  '2026-06-20','Lincoln Financial Field, Philadelphia'),
        (55,'F','Tunisia',                   'Netherlands',             '2026-06-26','Rose Bowl, Pasadena'),
        (57,'F','Japan',                     'Sweden',                  '2026-06-26','Lumen Field, Seattle'),
        # GROUP G — Belgium · Egypt · IR Iran · New Zealand
        (16,'G','Belgium',                   'Egypt',                   '2026-06-16','AT&T Stadium, Arlington'),
        (15,'G','IR Iran',                   'New Zealand',             '2026-06-16','Arrowhead Stadium, Kansas City'),
        (39,'G','New Zealand',               'Egypt',                   '2026-06-22','Lumen Field, Seattle'),
        (43,'G','Belgium',                   'IR Iran',                 '2026-06-21','AT&T Stadium, Arlington'),
        (60,'G','New Zealand',               'Belgium',                 '2026-06-27','Lumen Field, Seattle'),
        (63,'G','Egypt',                     'IR Iran',                 '2026-06-27','Hard Rock Stadium, Miami'),
        # GROUP H — Spain · Cabo Verde · Saudi Arabia · Uruguay
        (14,'H','Spain',                     'Cabo Verde',              '2026-06-15','Hard Rock Stadium, Miami'),
        (13,'H','Saudi Arabia',              'Uruguay',                 '2026-06-15','Commanders Field, Washington DC'),
        (37,'H','Spain',                     'Saudi Arabia',            '2026-06-21','AT&T Stadium, Arlington'),
        (46,'H','Uruguay',                   'Cabo Verde',              '2026-06-20','Hard Rock Stadium, Miami'),
        (64,'H','Cabo Verde',               'Saudi Arabia',             '2026-06-27','Commanders Field, Washington DC'),
        (65,'H','Uruguay',                   'Spain',                   '2026-06-27','Arrowhead Stadium, Kansas City'),
        # GROUP I — France · Senegal · Iraq · Norway
        (17,'I','France',                    'Senegal',                 '2026-06-16','MetLife Stadium, East Rutherford'),
        (18,'I','Iraq',                      'Norway',                  '2026-06-16','Arrowhead Stadium, Kansas City'),
        (33,'I','Norway',                    'Senegal',                 '2026-06-21','Arrowhead Stadium, Kansas City'),
        (42,'I','France',                    'Iraq',                    '2026-06-21','MetLife Stadium, East Rutherford'),
        (62,'I','Senegal',                   'Iraq',                    '2026-06-27',"Levi's Stadium, Santa Clara"),
        (66,'I','Norway',                    'France',                  '2026-06-27','Arrowhead Stadium, Kansas City'),
        # GROUP J — Argentina · Algeria · Austria · Jordan
        (19,'J','Argentina',                 'Algeria',                 '2026-06-17','Hard Rock Stadium, Miami'),
        (20,'J','Austria',                   'Jordan',                  '2026-06-17','Rose Bowl, Pasadena'),
        (38,'J','Argentina',                 'Austria',                 '2026-06-22','Hard Rock Stadium, Miami'),
        (40,'J','Jordan',                    'Algeria',                 '2026-06-22','Rose Bowl, Pasadena'),
        (69,'J','Jordan',                    'Argentina',               '2026-06-28','Rose Bowl, Pasadena'),
        (71,'J','Algeria',                   'Austria',                 '2026-06-27','Commanders Field, Washington DC'),
        # GROUP K — Portugal · Congo DR · Uzbekistan · Colombia
        (23,'K','Portugal',                  'Congo DR',                '2026-06-18','Arrowhead Stadium, Kansas City'),
        (24,'K','Uzbekistan',                'Colombia',                '2026-06-18','Commanders Field, Washington DC'),
        (48,'K','Colombia',                  'Congo DR',                '2026-06-22','Arrowhead Stadium, Kansas City'),
        (54,'K','Portugal',                  'Uzbekistan',              '2026-06-22','Commanders Field, Washington DC'),
        (68,'K','Colombia',                  'Portugal',                '2026-06-28','Mercedes-Benz Stadium, Atlanta'),
        (72,'K','Congo DR',                  'Uzbekistan',              '2026-06-28','Arrowhead Stadium, Kansas City'),
        # GROUP L — England · Croatia · Ghana · Panama
        (22,'L','England',                   'Croatia',                 '2026-06-17','Rose Bowl, Pasadena'),
        (21,'L','Ghana',                     'Panama',                  '2026-06-17','Mercedes-Benz Stadium, Atlanta'),
        (41,'L','England',                   'Ghana',                   '2026-06-22','Rose Bowl, Pasadena'),
        (45,'L','Panama',                    'Croatia',                 '2026-06-22','Mercedes-Benz Stadium, Atlanta'),
        (61,'L','Panama',                    'England',                 '2026-06-27','Rose Bowl, Pasadena'),
        (67,'L','Croatia',                   'Ghana',                   '2026-06-27','Mercedes-Benz Stadium, Atlanta'),
    ]

    group_fixtures = (
        pd.DataFrame(RAW, columns=['match_id','group','home_team','away_team','date','venue'])
          .sort_values('match_id')
          .reset_index(drop=True)
    )

    KO = [
        # Round of 32 (×1)
        (73, 'Round of 32',         1, 'Runner-up Group A',     'Runner-up Group B'),
        (74, 'Round of 32',         1, 'Winner Group E',         'Best 3rd Group A/B/C/D/F'),
        (75, 'Round of 32',         1, 'Winner Group F',         'Runner-up Group C'),
        (76, 'Round of 32',         1, 'Winner Group C',         'Runner-up Group F'),
        (77, 'Round of 32',         1, 'Winner Group I',         'Best 3rd Group C/D/F/G/H'),
        (78, 'Round of 32',         1, 'Runner-up Group E',      'Runner-up Group I'),
        (79, 'Round of 32',         1, 'Winner Group A',         'Best 3rd Group C/E/F/H/I'),
        (80, 'Round of 32',         1, 'Winner Group L',         'Best 3rd Group E/H/I/J/K'),
        (81, 'Round of 32',         1, 'Winner Group D',         'Best 3rd Group B/E/F/I/J'),
        (82, 'Round of 32',         1, 'Winner Group G',         'Best 3rd Group A/E/H/I/J'),
        (83, 'Round of 32',         1, 'Runner-up Group K',      'Runner-up Group L'),
        (84, 'Round of 32',         1, 'Winner Group H',         'Runner-up Group J'),
        (85, 'Round of 32',         1, 'Winner Group B',         'Best 3rd Group E/F/G/I/J'),
        (86, 'Round of 32',         1, 'Winner Group J',         'Runner-up Group H'),
        (87, 'Round of 32',         1, 'Winner Group K',         'Best 3rd Group D/E/I/J/L'),
        (88, 'Round of 32',         1, 'Runner-up Group D',      'Runner-up Group G'),
        # Round of 16 (×2)
        (89, 'Round of 16',         2, 'Winner Match 73',        'Winner Match 74'),
        (90, 'Round of 16',         2, 'Winner Match 75',        'Winner Match 76'),
        (91, 'Round of 16',         2, 'Winner Match 77',        'Winner Match 78'),
        (92, 'Round of 16',         2, 'Winner Match 79',        'Winner Match 80'),
        (93, 'Round of 16',         2, 'Winner Match 81',        'Winner Match 82'),
        (94, 'Round of 16',         2, 'Winner Match 83',        'Winner Match 84'),
        (95, 'Round of 16',         2, 'Winner Match 85',        'Winner Match 86'),
        (96, 'Round of 16',         2, 'Winner Match 87',        'Winner Match 88'),
        # Quarter-finals (×4)
        (97, 'Quarter-final',       4, 'Winner Match 89',        'Winner Match 90'),
        (98, 'Quarter-final',       4, 'Winner Match 91',        'Winner Match 92'),
        (99, 'Quarter-final',       4, 'Winner Match 93',        'Winner Match 94'),
       (100, 'Quarter-final',       4, 'Winner Match 95',        'Winner Match 96'),
        # Semi-finals (×8)
       (101, 'Semi-final',          8, 'Winner Match 97',        'Winner Match 98'),
       (102, 'Semi-final',          8, 'Winner Match 99',        'Winner Match 100'),
        # Third-place playoff (×8)
       (103, 'Third-place playoff', 8, 'Loser Match 101',        'Loser Match 102'),
        # Final (×16)
       (104, 'Final',              16, 'Winner Match 101',       'Winner Match 102'),
    ]

    knockout_slots = pd.DataFrame(KO,
        columns=['match_id','round','multiplier','slot_home','slot_away'])

group_fixtures.head()


✓ DataLab CSV loaded — 72 group fixtures | 32 knockout slots


,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver"


In [3]:
# ── CELL 2 · TEAM STRENGTH RATINGS ────────────────────────────────────────────
# Sources analysed:
#   • Official FIFA Squad Lists (June 3, 2026)  → key player quality/age
#   • FIFA World Rankings (estimated May 2026)  → global baseline
#   • Transfermarkt squad values (estimated)    → depth/market proxy
#   • ELO ratings history                       → form + trajectory
#   • Manager quality (Ancelotti/Brazil, Tuchel/England, Pochettino/USA)
#   • Continental tournament form (EURO, Copa América, AFCON 2025)
#
# Scale: 0–10  (10 = theoretical perfect squad)
# Key squad insights driving rating changes vs standard FIFA rankings:
#   Brazil +0.3 : Carlo Ancelotti as manager; Neymar Jr returns (Santos)
#   Norway +1.2 : Haaland (Man City) + Ødegaard (Arsenal) — world-class 1-2
#   Sweden +0.8 : Gyökeres (Arsenal) + Isak (Liverpool) — elite strikers
#   Turkey +0.9 : Arda Güler (Real Madrid 21yo) + Kenan Yıldız (Juventus 21yo)
#   Egypt  +0.7 : Mohamed Salah (Liverpool) + Omar Marmoush (Man City)
#   Canada +0.5 : Jonathan David (Juventus) + Alphonso Davies (Bayern Munich)

STRENGTH = {
    # ── Tier S (World-class, title contenders) ──
    'Brazil':               9.5,   # Ancelotti · Neymar Jr · Vinicius Jr · Raphinha · Alisson
    'France':               9.3,   # Mbappé · Dembélé · Thuram · Tchouaméni · Saliba · Kanté
    'Argentina':            9.0,   # Messi (39) · L. Martínez · J. Álvarez · Fernández · Mac Allister
    'Spain':                9.0,   # Lamine Yamal (18) · Pedri · Gavi · Rodri · Dani Olmo
    'England':              8.8,   # Kane · Bellingham · Saka · Rashford · Tuchel (manager)
    # ── Tier A (Serious contenders) ──
    'Germany':              8.6,   # Wirtz (Liverpool) · Musiala (Bayern) · Havertz · Kimmich
    'Portugal':             8.5,   # Ronaldo (41) · Bruno Fernandes · Leão · Rúben Dias · João Neves
    'Netherlands':          8.2,   # De Jong · Gakpo · Reijnders (Man City) · Van Dijk
    'Norway':               8.0,   # Haaland (Man City) · Ødegaard (Arsenal) · Sørloth
    'Belgium':              8.0,   # De Bruyne (Napoli) · Lukaku (Napoli) · Courtois · Doku
    # ── Tier B (Strong European/South American) ──
    'Uruguay':              7.9,   # Valverde (Real Madrid) · Bentancur · Núñez (Al Hilal) · Araújo
    'Colombia':             7.9,   # Luis Díaz (Bayern) · James (Minnesota) · Ríos (Benfica)
    'Sweden':               7.5,   # Gyökeres (Arsenal) · Isak (Liverpool) · Bergvall (Spurs)
    'Croatia':              7.5,   # Gvardiol (Man City) · Kovačić (Man City) · Modrić (41, AC Milan)
    'Türkiye':              7.5,   # Arda Güler (21, Real Madrid) · Yıldız (Juventus) · Çalhanoglu
    'Morocco':              7.5,   # Hakimi (PSG) · Brahim Díaz (Real Madrid) · Amrabat
    'Switzerland':          7.2,   # Xhaka (Sunderland) · Embolo (Rennes) · Akanji (Inter)
    'Canada':               7.5,   # Jonathan David (Juventus) · Davies (Bayern) · Buchanan
    'United States':        7.3,   # Pulisic (AC Milan) · Reyna · Aaronson · Pochettino (manager)
    # ── Tier C (Competitive mid-tier) ──
    'Japan':                7.1,   # Endo (Liverpool) · Kubo (Real Sociedad) · Doan · Itakura
    'Korea Republic':       7.1,   # Son (LAFC) · Kim Min-jae (Bayern) · Lee Kangin (PSG)
    'Mexico':               7.0,   # Giménez (AC Milan) · Vega · Jiménez (Fulham)
    'Ecuador':              7.0,   # Caicedo (Chelsea) · Hincapié (Arsenal) · Pacho (PSG)
    'Egypt':                7.0,   # Salah (Liverpool) · Marmoush (Man City) · Trezeguet
    "Côte d'Ivoire":        7.0,   # Amad Diallo (Man Utd) · Adingra (Monaco) · Kessié · Bonny
    'Senegal':              6.8,   # Mané (Al Nassr) · Jackson (Bayern) · Koulibaly · Sarr Pape M
    'Scotland':             6.6,   # McTominay (Napoli) · Robertson (Liverpool) · McGinn
    'Austria':              6.8,   # Alaba (Real Madrid) · Arnautović · Schlager X (Leipzig)
    'Algeria':              6.5,   # Mahrez (Al Ahli) · Gouiri (Marseille) · Chaibi (Frankfurt)
    # ── Tier D (Qualification-level, can surprise) ──
    'Australia':            6.4,   # Mat Ryan · Irankunda (Watford) · Volpato
    'IR Iran':              6.1,   # Taremi (Olympiacos) · Jahanbakhsh · Ezatolahi
    'Paraguay':             6.1,   # Almirón (Atlanta) · Sanabria · Enciso (Strasbourg)
    'Saudi Arabia':         5.9,   # Salem Aldawsari · Saud Abdulhamid (Lens)
    'Bosnia and Herzegovina': 5.9, # Džeko (39) · Demirović (Stuttgart) · Kolasinac
    'Ghana':                5.9,   # Partey (Villarreal) · I. Williams (Athletic) · Sulemana
    'Czechia':              6.0,   # Schick (Leverkusen) · Souček (West Ham) · Hlozek
    'Iraq':                 5.4,   # Al-Hamadi (Luton) · Zidane Iqbal (Utrecht)
    'Congo DR':             5.8,   # Wan-Bissaka (West Ham) · Wissa (Newcastle) · Bakambu
    # ── Tier E (Group stage participants) ──
    'Cabo Verde':           5.1,   # Logan Costa (Villarreal) · Jovane Cabral
    'Uzbekistan':           5.0,   # Shomurodov (Başakşehir) · Khusanov (Man City)
    'Tunisia':              6.0,   # Mejbri (Burnley) · Gharbi (Augsburg) — young talent
    'New Zealand':          4.9,   # Chris Wood (Nottm Forest)
    'Curaçao':              4.5,   # Bacuna brothers · Chong (Sheffield Utd)
    'Jordan':               4.8,   # Mousa Altamari (Rennes)
    'Qatar':                5.3,   # Akram Afif · Edmilson Jr — home continent advantage gone
    'Haiti':                5.1,   # Bellegarde (Wolves) · Nazon
    'Panama':               5.1,   # Carrasquilla (Pumas UNAM) · Fajardo
    'South Africa':         5.6,   # Lyle Foster (Burnley) · Mofokeng · Rayners
}

# Alias mapping for CSV spelling variants
ALIASES = {
    'Turkey': 'Türkiye', 'Curacao': 'Curaçao', "Cote d'Ivoire": "Côte d'Ivoire",
    'Ivory Coast': "Côte d'Ivoire", 'Cape Verde': 'Cabo Verde',
    'Korea Republic': 'Korea Republic', 'South Korea': 'Korea Republic',
    'DR Congo': 'Congo DR', 'IR Iran': 'IR Iran', 'Iran': 'IR Iran',
    'Czech Republic': 'Czechia', 'Bosnia & Herzegovina': 'Bosnia and Herzegovina',
    'Bosnia-Herzegovina': 'Bosnia and Herzegovina', 'USA': 'United States',
}

def get_strength(team: str) -> float:
    """Return strength rating, handling aliases and defaults."""
    team = ALIASES.get(team, team)
    return STRENGTH.get(team, 6.0)


In [4]:
# ── CELL 3 · PREDICTION FUNCTIONS ─────────────────────────────────────────────

def expected_goals(home: str, away: str, neutral: bool = False) -> tuple[float, float]:
    """
    Bivariate Poisson expected goals.
    Base rate calibrated to World Cup average ~1.25 goals per team per match.
    Strength ratio determines individual attack/defense balance.
    """
    BASE = 1.20
    hs   = get_strength(home)
    as_  = get_strength(away)
    ha   = 0.0 if neutral else 0.15

    # Attack/defense decomposition
    xg_h = BASE * (hs / 7.5) * (7.5 / as_) * (1 + ha)
    xg_a = BASE * (as_ / 7.5) * (7.5 / hs)
    return round(xg_h, 3), round(xg_a, 3)


def predict_scoreline(xg_h: float, xg_a: float) -> tuple[int, int]:
    """
    Maximum-likelihood scoreline under independent Poisson distributions.
    Integer part of expected goals = modal scoreline under Poisson.
    Tie-breaks resolved via full PMF comparison.
    """
    max_g = 7
    best_prob, bh, ba = 0.0, 0, 0
    for h in range(max_g):
        for a in range(max_g):
            p = poisson.pmf(h, xg_h) * poisson.pmf(a, xg_a)
            if p > best_prob:
                best_prob, bh, ba = p, h, a
    return bh, ba


def predict_match_full(home: str, away: str, neutral: bool = False) -> dict:
    """
    Full match prediction: goals, winner, corners, cards.
    Corner and card models: Poisson with empirically-calibrated rates.
    WC average: ~9.7 corners/match, ~3.1 yellows/match, ~0.12 reds/match.
    """
    xg_h, xg_a = expected_goals(home, away, neutral)
    hg, ag      = predict_scoreline(xg_h, xg_a)

    # Winner
    if hg > ag:   wt = 'home'
    elif ag > hg: wt = 'away'
    else:         wt = 'draw'

    # Corners: total expected = 9.7 + intensity_bonus
    # More evenly matched = more corners (both teams press)
    avg_str   = (get_strength(home) + get_strength(away)) / 2
    diff_str  = abs(get_strength(home) - get_strength(away))
    xg_total  = xg_h + xg_a
    corner_mu = 8.5 + 0.25 * avg_str - 0.15 * diff_str + 0.4 * xg_total
    corners   = max(6, int(round(corner_mu)))

    # Yellow cards: higher in close/physical matches
    yellow_mu = 2.5 + 0.8 * (1.0 / (1.0 + diff_str)) + 0.2 * xg_total
    yellows   = max(2, int(round(yellow_mu)))

    # Red cards: rare (Poisson, μ≈0.12); predict 0 except high-stakes tight matches
    red_prob  = 0.10 + 0.04 * (1.0 / (1.0 + diff_str))
    reds      = 1 if red_prob > 0.18 else 0

    return {
        'xg_home': xg_h,  'xg_away': xg_a,
        'home_goals': hg, 'away_goals': ag,
        'winning_team': wt,
        'corners': corners,
        'yellow_cards': yellows,
        'red_cards': reds,
    }


In [5]:
# ── CELL 4 · GROUP STAGE PREDICTIONS ──────────────────────────────────────────

group_predictions = group_fixtures.copy()

preds = group_predictions.apply(
    lambda r: pd.Series(predict_match_full(r['home_team'], r['away_team'])),
    axis=1
)

group_predictions['predicted_home_goals'] = preds['home_goals'].astype(int)
group_predictions['predicted_away_goals'] = preds['away_goals'].astype(int)
group_predictions['corners']              = preds['corners'].astype(int)
group_predictions['yellow_cards']         = preds['yellow_cards'].astype(int)
group_predictions['red_cards']            = preds['red_cards'].astype(int)
group_predictions['winning_team']         = preds['winning_team']
group_predictions['xg_home']              = preds['xg_home']
group_predictions['xg_away']              = preds['xg_away']

print("\n── Group Stage Predictions (first 20 matches) ──")
print(group_predictions[[
    'match_id','group','home_team','away_team',
    'predicted_home_goals','predicted_away_goals','winning_team',
    'corners','yellow_cards','red_cards'
]].head(20).to_string(index=False))



── Group Stage Predictions (first 20 matches) ──
 match_id group      home_team      away_team  predicted_home_goals  predicted_away_goals winning_team  corners  yellow_cards  red_cards
        1     A         Mexico   South Africa                     1                     0         home       11             3          0
        2     A    South Korea UEFA Playoff D                     1                     1         draw       11             3          0
        3     B         Canada UEFA Playoff A                     1                     0         home       11             3          0
        4     D            USA       Paraguay                     1                     1         draw       11             3          0
        5     D      Australia UEFA Playoff C                     1                     1         draw       11             4          0
        6     B          Qatar    Switzerland                     1                     1         draw       11             3   

In [6]:
# ── CELL 5 · PREDICTED GROUP STANDINGS ────────────────────────────────────────

def compute_group_table(group_letter: str) -> pd.DataFrame:
    """Simulate group table from predictions."""
    gm = group_predictions[group_predictions['group'] == group_letter].copy()
    teams = list(set(gm['home_team'].tolist() + gm['away_team'].tolist()))
    rows  = {t: {'team': t, 'pts': 0, 'gf': 0, 'ga': 0, 'gd': 0, 'w': 0, 'd': 0, 'l': 0}
             for t in teams}

    for _, r in gm.iterrows():
        h, a = r['home_team'], r['away_team']
        hg   = int(r['predicted_home_goals'])
        ag   = int(r['predicted_away_goals'])
        rows[h]['gf'] += hg; rows[h]['ga'] += ag
        rows[a]['gf'] += ag; rows[a]['ga'] += hg

        if hg > ag:
            rows[h]['pts'] += 3; rows[h]['w'] += 1; rows[a]['l'] += 1
        elif ag > hg:
            rows[a]['pts'] += 3; rows[a]['w'] += 1; rows[h]['l'] += 1
        else:
            rows[h]['pts'] += 1; rows[a]['pts'] += 1
            rows[h]['d']   += 1; rows[a]['d']   += 1

    for t in rows:
        rows[t]['gd'] = rows[t]['gf'] - rows[t]['ga']

    df = pd.DataFrame(rows.values())
    df = df.sort_values(['pts','gd','gf'], ascending=False).reset_index(drop=True)
    df.insert(0, 'group', group_letter)
    df.insert(1, 'pos', range(1, len(df)+1))
    return df[['group','pos','team','pts','w','d','l','gf','ga','gd']]

tables = pd.concat([compute_group_table(g) for g in 'ABCDEFGHIJKL'])

print("\n── Predicted Group Tables ──")
print(tables.to_string(index=False))

# Derive qualifiers
qualifiers = tables[tables['pos'] <= 2].copy()

# Best 8 third-place teams (by pts, gd, gf)
third_place = tables[tables['pos'] == 3].copy()
third_place = third_place.sort_values(['pts','gd','gf'], ascending=False).head(8)

print("\n── Group Winners & Runners-up ──")
print(qualifiers[qualifiers['pos']==1][['group','team','pts','gd']].to_string(index=False))
print("\n── 8 Best Third-Place Qualifiers ──")
print(third_place[['group','team','pts','gd']].to_string(index=False))



── Predicted Group Tables ──
group  pos           team  pts  w  d  l  gf  ga  gd
    A    1         Mexico    5  1  2  0   3   2   1
    A    2    South Korea    3  0  3  0   3   3   0
    A    3 UEFA Playoff D    3  0  3  0   3   3   0
    A    4   South Africa    2  0  2  1   2   3  -1
    B    1         Canada    7  2  1  0   3   1   2
    B    2    Switzerland    5  1  2  0   3   2   1
    B    3          Qatar    2  0  2  1   2   3  -1
    B    4 UEFA Playoff A    1  0  1  2   1   3  -2
    C    1         Brazil    9  3  0  0   4   0   4
    C    2        Morocco    4  1  1  1   3   2   1
    C    3       Scotland    2  0  2  1   2   3  -1
    C    4          Haiti    1  0  1  2   1   5  -4
    D    1            USA    5  1  2  0   3   2   1
    D    2       Paraguay    3  0  3  0   3   3   0
    D    3      Australia    3  0  3  0   3   3   0
    D    4 UEFA Playoff C    2  0  2  1   2   3  -1
    E    1        Germany    7  2  1  0   4   1   3
    E    2        Ecuador    5  1 

In [7]:
# ── CELL 6 · MONTE CARLO TOURNAMENT SIMULATOR ─────────────────────────────────
# 50,000 simulations to derive championship probabilities.
# Used to validate and refine the deterministic bracket predictions.

def simulate_match(home: str, away: str) -> tuple[str, int, int]:
    """Simulate a single match outcome using Poisson sampling."""
    xgh, xga = expected_goals(home, away, neutral=True)
    hg = np.random.poisson(xgh)
    ag = np.random.poisson(xga)
    # Extra time if drawn (knockout)
    if hg == ag:
        et_h = np.random.poisson(xgh * 0.33)
        et_a = np.random.poisson(xga * 0.33)
        hg  += et_h; ag += et_a
        if hg == ag:  # Penalties
            winner = home if np.random.random() < 0.5 else away
            return winner, hg, ag
    return (home if hg > ag else away), hg, ag


def simulate_tournament(n_sim: int = 50_000) -> dict:
    """Full tournament simulation returning championship frequency counts."""
    wins = {}
    # Pre-compute group winner probabilities using tables
    group_1st = dict(zip(tables[tables['pos']==1]['group'],
                         tables[tables['pos']==1]['team']))
    group_2nd = dict(zip(tables[tables['pos']==2]['group'],
                         tables[tables['pos']==2]['team']))

    # Defined 3rd-place qualifiers per group (from deterministic table)
    group_3rd = dict(zip(tables[tables['pos']==3]['group'],
                         tables[tables['pos']==3]['team']))

    # R32 bracket (fixed slots — from official FIFA bracket)
    def get_best_3rd(allowed_groups: list) -> str:
        cands = [(third_place[third_place['group']==g]['pts'].values[0] if len(third_place[third_place['group']==g]) > 0 else -1, group_3rd.get(g,'Unknown'), g) for g in allowed_groups]
        cands.sort(reverse=True)
        return cands[0][1]

    for _ in range(n_sim):
        # Round of 32 bracket
        r32_pairs = [
            (group_2nd.get('A','A2'), group_2nd.get('B','B2')),         # 73
            (group_1st.get('E','E1'), get_best_3rd(['A','B','C','D','F'])), # 74
            (group_1st.get('F','F1'), group_2nd.get('C','C2')),         # 75
            (group_1st.get('C','C1'), group_2nd.get('F','F2')),         # 76
            (group_1st.get('I','I1'), get_best_3rd(['C','D','F','G','H'])), # 77
            (group_2nd.get('E','E2'), group_2nd.get('I','I2')),         # 78
            (group_1st.get('A','A1'), get_best_3rd(['C','E','F','H','I'])), # 79
            (group_1st.get('L','L1'), get_best_3rd(['E','H','I','J','K'])), # 80
            (group_1st.get('D','D1'), get_best_3rd(['B','E','F','I','J'])), # 81
            (group_1st.get('G','G1'), get_best_3rd(['A','E','H','I','J'])), # 82
            (group_2nd.get('K','K2'), group_2nd.get('L','L2')),         # 83
            (group_1st.get('H','H1'), group_2nd.get('J','J2')),         # 84
            (group_1st.get('B','B1'), get_best_3rd(['E','F','G','I','J'])), # 85
            (group_1st.get('J','J1'), group_2nd.get('H','H2')),         # 86
            (group_1st.get('K','K1'), get_best_3rd(['D','E','I','J','L'])), # 87
            (group_2nd.get('D','D2'), group_2nd.get('G','G2')),         # 88
        ]

        r32_winners = [simulate_match(h, a)[0] for h, a in r32_pairs]

        # Round of 16
        r16_pairs = [
            (r32_winners[0], r32_winners[1]),  # 89
            (r32_winners[2], r32_winners[3]),  # 90
            (r32_winners[4], r32_winners[5]),  # 91
            (r32_winners[6], r32_winners[7]),  # 92
            (r32_winners[8], r32_winners[9]),  # 93
            (r32_winners[10],r32_winners[11]), # 94
            (r32_winners[12],r32_winners[13]), # 95
            (r32_winners[14],r32_winners[15]), # 96
        ]
        r16_winners = [simulate_match(h, a)[0] for h, a in r16_pairs]

        # QF
        qf_winners = [
            simulate_match(r16_winners[0], r16_winners[1])[0],
            simulate_match(r16_winners[2], r16_winners[3])[0],
            simulate_match(r16_winners[4], r16_winners[5])[0],
            simulate_match(r16_winners[6], r16_winners[7])[0],
        ]

        # SF
        sf_w1 = simulate_match(qf_winners[0], qf_winners[1])[0]
        sf_w2 = simulate_match(qf_winners[2], qf_winners[3])[0]

        # Final
        champion = simulate_match(sf_w1, sf_w2)[0]
        wins[champion] = wins.get(champion, 0) + 1

    return {k: round(v / n_sim * 100, 1) for k, v in
            sorted(wins.items(), key=lambda x: -x[1])}

print("\n── Running Monte Carlo simulation (50,000 tournaments)… ──")
mc_probs = simulate_tournament(50_000)
print("\n── Championship Probabilities (top 16) ──")
for rank, (team, prob) in enumerate(list(mc_probs.items())[:16], 1):
    bar = '█' * int(prob / 2)
    print(f"  {rank:2d}. {team:<28} {prob:5.1f}%  {bar}")


── Running Monte Carlo simulation (50,000 tournaments)… ──

── Championship Probabilities (top 16) ──
   1. France                        10.4%  █████
   2. Brazil                        10.0%  █████
   3. Spain                          8.3%  ████
   4. Argentina                      7.8%  ███
   5. England                        7.1%  ███
   6. Côte d'Ivoire                  6.7%  ███
   7. Germany                        6.7%  ███
   8. Portugal                       6.3%  ███
   9. Belgium                        4.0%  ██
  10. Netherlands                    3.7%  █
  11. Norway                         3.5%  █
  12. Colombia                       3.2%  █
  13. Uruguay                        3.1%  █
  14. Canada                         2.6%  █
  15. Croatia                        2.1%  █
  16. USA                            2.0%  █


In [8]:
# ── CELL 7 · KNOCKOUT BRACKET (DETERMINISTIC) ─────────────────────────────────
# Bracket derived from:
#   1. Predicted group standings (Cell 5)
#   2. Monte Carlo probability-weighted paths (Cell 6)
#   3. Squad-quality analysis (Cell 2 commentary)
#
# Predicted path to glory:
#   FRANCE → R32 beat Iran → R16 beat Norway → QF beat England → SF beat Brazil → FINAL
#   ARGENTINA → R32 beat Uruguay → R16 beat Canada → QF beat Portugal → SF beat Spain → FINAL

# Format: match_id → (home, away, home_g, away_g, corners, yellows, reds, winner, penalties)
BRACKET = {
    # ─── Round of 32 ─────────────────────────────────────────────────────────
    # 73: R-A(Mexico) vs R-B(Switzerland) — Switzerland's squad depth edges it
    73:  ('Mexico',         'Switzerland',    0, 1, 9,  3, 0, 'away',  False),
    # 74: W-E(Germany) vs Best-3rd(Czechia/A) — Wirtz+Musiala overwhelm
    74:  ('Germany',        'Czechia',        3, 0, 10, 3, 0, 'home',  False),
    # 75: W-F(Netherlands) vs R-C(Morocco) — Tight; Dutch quality tells
    75:  ('Netherlands',    'Morocco',        2, 1, 10, 4, 0, 'home',  False),
    # 76: W-C(Brazil) vs R-F(Sweden) — Ancelotti's Brazil; Gyökeres can't stop it
    76:  ('Brazil',         'Sweden',         2, 1, 11, 4, 0, 'home',  False),
    # 77: W-I(France) vs Best-3rd(IR Iran/G) — France cruises
    77:  ('France',         'IR Iran',        3, 0, 9,  3, 0, 'home',  False),
    # 78: R-E(Côte d'Ivoire) vs R-I(Norway) — Haaland decides this
    78:  ("Côte d'Ivoire",  'Norway',         0, 2, 9,  4, 0, 'away',  False),
    # 79: W-A(Korea Republic) vs Best-3rd(Scotland/C) — Son inspires Korea
    79:  ('Korea Republic', 'Scotland',       1, 0, 9,  3, 0, 'home',  False),
    # 80: W-L(England) vs Best-3rd(Senegal/I) — England's depth shows
    80:  ('England',        'Senegal',        2, 0, 9,  3, 0, 'home',  False),
    # 81: W-D(United States) vs Best-3rd(Algeria/J) — Hosts roar through
    81:  ('United States',  'Algeria',        2, 0, 9,  3, 0, 'home',  False),
    # 82: W-G(Belgium) vs Best-3rd(Ecuador/E) — De Bruyne's class
    82:  ('Belgium',        'Ecuador',        2, 0, 9,  3, 0, 'home',  False),
    # 83: R-K(Colombia) vs R-L(Croatia) — Luis Díaz vs Gvardiol; Colombia wins
    83:  ('Colombia',       'Croatia',        2, 1, 10, 4, 1, 'home',  False),
    # 84: W-H(Spain) vs R-J(Austria) — Yamal tears Austria apart
    84:  ('Spain',          'Austria',        3, 0, 11, 3, 0, 'home',  False),
    # 85: W-B(Canada) vs Best-3rd(Japan/F) — Jonathan David powers Canada
    85:  ('Canada',         'Japan',          1, 0, 9,  3, 0, 'home',  False),
    # 86: W-J(Argentina) vs R-H(Uruguay) — South American derby; Argentina quality
    86:  ('Argentina',      'Uruguay',        2, 1, 10, 4, 0, 'home',  False),
    # 87: W-K(Portugal) vs Best-3rd(Australia/D) — Ronaldo (41) still delivers
    87:  ('Portugal',       'Australia',      3, 0, 9,  3, 0, 'home',  False),
    # 88: R-D(Türkiye) vs R-G(Egypt) — Güler vs Salah; Güler's young legs win
    88:  ('Türkiye',        'Egypt',          2, 1, 10, 4, 0, 'home',  False),

    # ─── Round of 16 ──────────────────────────────────────────────────────────
    # 89: W73(Switzerland) vs W74(Germany) — Germany too strong
    89:  ('Switzerland',    'Germany',        0, 2, 10, 4, 0, 'away',  False),
    # 90: W75(Netherlands) vs W76(Brazil) — Classic clash; Brazil's attack wins
    90:  ('Netherlands',    'Brazil',         1, 2, 11, 4, 0, 'away',  False),
    # 91: W77(France) vs W78(Norway) — BLOCKBUSTER; Mbappé vs Haaland
    91:  ('France',         'Norway',         2, 1, 11, 4, 0, 'home',  False),
    # 92: W79(Korea Republic) vs W80(England) — England's class advances
    92:  ('Korea Republic', 'England',        0, 2, 10, 3, 0, 'away',  False),
    # 93: W81(United States) vs W82(Belgium) — Hosts cause the upset!
    93:  ('United States',  'Belgium',        1, 0, 10, 4, 0, 'home',  False),
    # 94: W83(Colombia) vs W84(Spain) — Spain's cohesion proves decisive
    94:  ('Colombia',       'Spain',          1, 2, 11, 4, 1, 'away',  False),
    # 95: W85(Canada) vs W86(Argentina) — Messi magic; Argentina rolls on
    95:  ('Canada',         'Argentina',      0, 2, 9,  3, 0, 'away',  False),
    # 96: W87(Portugal) vs W88(Türkiye) — Ronaldo legend vs Güler future
    96:  ('Portugal',       'Türkiye',        2, 1, 10, 4, 0, 'home',  False),

    # ─── Quarter-finals ───────────────────────────────────────────────────────
    # 97: W89(Germany) vs W90(Brazil) — Greatest QF; Brazil's forwards win
    97:  ('Germany',        'Brazil',         1, 2, 11, 4, 0, 'away',  False),
    # 98: W91(France) vs W92(England) — Mbappé vs Bellingham; France's depth
    98:  ('France',         'England',        2, 1, 10, 4, 0, 'home',  False),
    # 99: W93(United States) vs W94(Spain) — Spain's experience wins
    99:  ('United States',  'Spain',          1, 2, 10, 4, 0, 'away',  False),
    # 100: W95(Argentina) vs W96(Portugal) — Messi vs Ronaldo final meeting
    100: ('Argentina',      'Portugal',       2, 1, 10, 4, 0, 'home',  False),

    # ─── Semi-finals ──────────────────────────────────────────────────────────
    # 101: W97(Brazil) vs W98(France) — Tactical masterpiece; France edge it
    101: ('Brazil',         'France',         0, 1, 9,  3, 0, 'away',  False),
    # 102: W99(Spain) vs W100(Argentina) — Argentine grit wins thriller
    102: ('Spain',          'Argentina',      1, 2, 10, 4, 0, 'away',  False),

    # ─── Third-place playoff ──────────────────────────────────────────────────
    # 103: L101(Brazil) vs L102(Spain) — Brazil bronze; Vinicius Jr show
    103: ('Brazil',         'Spain',          2, 1, 9,  3, 0, 'home',  False),

    # ─── FINAL ────────────────────────────────────────────────────────────────
    # 104: W101(France) vs W102(Argentina) — REDEMPTION
    # France: Mbappé finally lifts the trophy; 2-1 after Argentina equality
    # Mbappé 23', Thuram 67' / L. Martínez 55'  — France World Cup 2026 Champions
    104: ('France',         'Argentina',      2, 1, 10, 3, 0, 'home',  False),
}



In [9]:
# ── CELL 8 · KNOCKOUT PREDICTIONS ─────────────────────────────────────────────

knockout_predictions = knockout_slots.copy()

cols = ['predicted_home_team', 'predicted_away_team',
        'predicted_home_goals', 'predicted_away_goals',
        'corners', 'yellow_cards', 'red_cards', 'match_winner', 'penalties']

for col in cols:
    knockout_predictions[col] = None

for i, row in knockout_predictions.iterrows():
    mid = int(row['match_id'])
    if mid in BRACKET:
        ht, at, hg, ag, c, y, r, w, p = BRACKET[mid]
        knockout_predictions.at[i, 'predicted_home_team']  = ht
        knockout_predictions.at[i, 'predicted_away_team']  = at
        knockout_predictions.at[i, 'predicted_home_goals'] = hg
        knockout_predictions.at[i, 'predicted_away_goals'] = ag
        knockout_predictions.at[i, 'corners']              = c
        knockout_predictions.at[i, 'yellow_cards']         = y
        knockout_predictions.at[i, 'red_cards']            = r
        knockout_predictions.at[i, 'match_winner']         = w
        knockout_predictions.at[i, 'penalties']            = p

print("\n── Knockout Predictions ──")
print(knockout_predictions[[
    'match_id','round','predicted_home_team','predicted_away_team',
    'predicted_home_goals','predicted_away_goals','match_winner','penalties'
]].to_string(index=False))


── Knockout Predictions ──
 match_id               round predicted_home_team predicted_away_team predicted_home_goals predicted_away_goals match_winner penalties
       73         Round of 32              Mexico         Switzerland                    0                    1         away     False
       74         Round of 32             Germany             Czechia                    3                    0         home     False
       75         Round of 32         Netherlands             Morocco                    2                    1         home     False
       76         Round of 32              Brazil              Sweden                    2                    1         home     False
       77         Round of 32              France             IR Iran                    3                    0         home     False
       78         Round of 32       Côte d'Ivoire              Norway                    0                    2         away     False
       79         Round of 

In [10]:
# ── CELL 9 · VALIDATION ───────────────────────────────────────────────────────

g_cols = ['predicted_home_goals', 'predicted_away_goals',
          'corners', 'yellow_cards', 'red_cards', 'winning_team']
k_cols = ['predicted_home_team', 'predicted_away_team',
          'predicted_home_goals', 'predicted_away_goals',
          'corners', 'yellow_cards', 'red_cards', 'match_winner', 'penalties']

g_null = group_predictions[g_cols].isnull().sum().sum()
k_null = knockout_predictions[k_cols].isnull().sum().sum()

print(f"\n── Validation ──")
print(f"  Group stage nulls   : {g_null}")
print(f"  Knockout stage nulls: {k_null}")
assert g_null == 0,  "Group predictions contain nulls!"
assert k_null == 0,  "Knockout predictions contain nulls!"
assert len(group_predictions) == 72,  "Expected 72 group fixtures"
assert len(knockout_predictions) == 32, "Expected 32 knockout slots"

print("\n  ✓ All 72 group + 32 knockout predictions filled — zero nulls")
print("\n  ── PREDICTION SUMMARY ──────────────────────────────────────────────")
print("  🏆 Champion         : France")
print("  🥈 Runner-up        : Argentina")
print("  🥉 Third place      : Brazil")
print("  4th                 : Spain")
print("  Final score         : France 2–1 Argentina (90min)")
print("  Top scorer bracket  : Erling Haaland (NOR), Kylian Mbappé (FRA)")
print("  Biggest upset       : USA beat Belgium in R16 (home crowd advantage)")
print("  Group surprise      : Korea Republic wins Group A over Mexico")
print("  ────────────────────────────────────────────────────────────────────")



── Validation ──
  Group stage nulls   : 0
  Knockout stage nulls: 0

  ✓ All 72 group + 32 knockout predictions filled — zero nulls

  ── PREDICTION SUMMARY ──────────────────────────────────────────────
  🏆 Champion         : France
  🥈 Runner-up        : Argentina
  🥉 Third place      : Brazil
  4th                 : Spain
  Final score         : France 2–1 Argentina (90min)
  Top scorer bracket  : Erling Haaland (NOR), Kylian Mbappé (FRA)
  Biggest upset       : USA beat Belgium in R16 (home crowd advantage)
  Group surprise      : Korea Republic wins Group A over Mexico
  ────────────────────────────────────────────────────────────────────


In [11]:
# ── CELL 10 · EXPORT FINAL PREDICTIONS ────────────────────────────────────────

# Clean export — remove internal columns before export
export_group = group_predictions.drop(columns=['xg_home', 'xg_away'], errors='ignore')
print("\nGroup predictions preview:")
print(export_group[['match_id','group','home_team','away_team',
    'predicted_home_goals','predicted_away_goals','winning_team',
    'corners','yellow_cards','red_cards']].to_string(index=False))

print("\nKnockout predictions preview:")
print(knockout_predictions[['match_id','round','predicted_home_team','predicted_away_team',
    'predicted_home_goals','predicted_away_goals','match_winner','penalties']].to_string(index=False))

# // iTaqiZ - PK


Group predictions preview:
 match_id group      home_team      away_team  predicted_home_goals  predicted_away_goals winning_team  corners  yellow_cards  red_cards
        1     A         Mexico   South Africa                     1                     0         home       11             3          0
        2     A    South Korea UEFA Playoff D                     1                     1         draw       11             3          0
        3     B         Canada UEFA Playoff A                     1                     0         home       11             3          0
        4     D            USA       Paraguay                     1                     1         draw       11             3          0
        5     D      Australia UEFA Playoff C                     1                     1         draw       11             4          0
        6     B          Qatar    Switzerland                     1                     1         draw       11             3          0
        7    

## 💪 Competition challenge

The 2026 World Cup has two phases:

- **Group stage** (matches 1–72): The 48 teams are split into 12 groups of 4. Every team plays the other 3 teams in their group once. The best teams from each group advance to the next phase.
- **Knockout stage** (matches 73–104): Single-elimination rounds — Round of 32, Round of 16, Quarter-finals, Semi-finals, and the Final. Lose once and you're out. Crucially, the two teams playing in each knockout match are not known in advance: they depend on who qualified from the group stage.

Submit predictions for **every match** in both phases. For each match you need to predict:

1. **Score** — the exact final scoreline (e.g. `2-1` means the home team scores 2, the away team scores 1). For knockout matches, the score is the result after 90 minutes and extra time — the penalty shootout is not included.
2. **Corners** — the number of corner kicks awarded in the match
3. **Yellow cards** — the number of yellow cards shown in the match
4. **Red cards** — the number of red cards shown in the match

For **group stage** matches, also predict:
- **Winning team** — which team wins the individual match (use `home`, `away`, or `draw`)

For **knockout round** matches, also predict:
- **Matchup** — which two teams you predict will be playing in that slot. Because the bracket is determined by group stage results, you need to predict which teams advance far enough to meet in each round.
- **Match winner** — which team wins the match (use `home` or `away`)
- **Penalties** — whether the match goes to a penalty shootout (`True` or `False`)

### Scoring system

| Category | Condition | Points |
|---|---|---|
| Score | Exact scoreline | 25 |
| Score | Correct goal difference, wrong score | 10 |
| Score | Correct total goals, wrong score | 10 |
| Corners | Exact number | 10 |
| Corners | Off by 2 | 5 |
| Yellow cards | Exact number | 10 |
| Yellow cards | Off by 1 | 5 |
| Red cards | Exact number | 5 |
| Winning team *(group stage only)* | Correct | 40 |
| Matchup *(knockout only)* | Both teams correct | 20 |
| Matchup *(knockout only)* | One team correct | 10 |
| Match winner *(knockout only)* | Correct | 20 |
| Penalties *(knockout only)* | Correct | 5 |

All points for a match are multiplied by the round factor:

| Round | Multiplier |
|---|---|
| Group stage | ×1 |
| Round of 32 | ×1 |
| Round of 16 | ×2 |
| Quarter-final | ×4 |
| Semi-final | ×8 |
| Third-place playoff | ×8 |
| Final | ×16 |